#Desenvolvendo um Sistema de RH com SQLAlchemy

In [19]:
import os
import pandas as pd
from sqlalchemy import Column, Integer, String, Float, MetaData, Table, and_, or_, create_engine, delete, insert, func, select, text, update, ForeignKey
from sqlalchemy.orm import declarative_base, mapped_column, relationship, sessionmaker

##Nível 1: Básico — Configuração e SQL Puro com Segurança

**Passo 1**: Crie a conexão com um banco de dados local utilizando a função create_engine('sqlite:///sistema_rh.db').

In [2]:
if os.path.exists('sistema_rh.db'):
    os.remove('sistema_rh.db')
engine = create_engine('sqlite:///sistema_rh.db')
print('Engine pronta! Banco: ', engine.url.database)

Engine pronta! Banco:  sistema_rh.db


**Passo 2**: Abra uma transação com with engine.begin() as conn: e utilize a função text() para executar um comando CREATE TABLE em SQL puro, criando uma tabela chamada funcionarios (com id, nome, cargo e salario).

In [3]:
with engine.begin() as conn:
    conn.execute(text('''
      CREATE TABLE funcionarios (id INTEGER PRIMARY KEY, nome TEXT, cargo TEXT, salario REAL)
    '''))

print('Tabela "funcionario" criada!')

Tabela "funcionario" criada!


**Passo 3 (Segurança)**: Simule a inserção de um novo funcionário a partir de um formulário web. Utilize o comando INSERT passando parâmetros seguros (ex: :nome, :cargo) e um dicionário de valores correspondentes. Pergunta reflexiva para os alunos: Por que nunca devemos concatenar strings diretamente no SQL (risco de injeção SQL) e como os placeholders resolvem isso?<br>

In [12]:
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO funcionarios (nome, cargo, salario) VALUES (:nome, :cargo, :salario)"),
        [
            {"nome": "João", "cargo": "Desenvolvedor Júnior", "salario": 5000.0},
            {"nome": "Maria", "cargo": "Gerente", "salario": 7000.0},
            {"nome": "Pedro", "cargo": "Analista", "salario": 4000.0}
        ]
    )

print('Funcionários inseridos!')

Funcionários inseridos!


**Passo 4**: Valide a inserção consultando os dados com a função pd.read_sql_query(), que já retorna a tabela formatada diretamente como um DataFrame do Pandas.


In [13]:
df_funcionarios = pd.read_sql_query(text('SELECT * FROM funcionarios'), engine.connect())
df_funcionarios

,id,nome,cargo,salario
0,1,João,Desenvolvedor,5000.0
1,2,Maria,Gerente,7000.0
2,3,Pedro,Analista,4000.0
3,4,João,Desenvolvedor Júnior,5000.0
4,5,Maria,Gerente,7000.0
5,6,Pedro,Analista,4000.0


##Nível 2: Intermediário — SQLAlchemy Core (Automatização Programática)

**Passo 1**: Defina uma nova tabela chamada projetos de forma programática utilizando os objetos Table, MetaData e Column. Em seguida, crie a tabela fisicamente no banco com metadata.create_all(engine).

In [14]:
metadata = MetaData()

projetos = Table(
    'projetos', metadata,
    Column('id', Integer, primary_key=True),
    Column('nome', String(50)),
    Column('descricao', String(200)),
    Column('orcamento', Float)
)

metadata.create_all(engine)
print('Tabela "projetos" criada!')

Tabela "projetos" criada!


**Passo 2**: Receba uma lista de dicionários contendo múltiplos projetos e faça uma inserção em lote (bulk insert) passando essa lista para o comando conn.execute(insert(projetos), [lista_de_dicts]).

In [15]:
registros = [
    {'nome': 'Projeto A', 'descricao': 'Descrição do Projeto A', 'orcamento': 100000.0},
    {'nome': 'Projeto B', 'descricao': 'Descrição do Projeto B', 'orcamento': 75000.0},
    {'nome': 'Projeto C', 'descricao': 'Descrição do Projeto C', 'orcamento': 50000.0}
]

with engine.connect() as conn:
    conn.execute(insert(projetos), registros)

print('Projetos inseridos!')

Projetos inseridos!


**Passo 3**: A diretoria aprovou um reajuste. Utilize a instrução update(tabela).where(...).values(...) para aumentar o salário apenas dos funcionários que ocupam o cargo de 'Desenvolvedor Júnior'

In [16]:
funcionarios = Table('funcionarios', metadata, autoload_with=engine)
with engine.connect() as conn:
    conn.execute(

        update(funcionarios).
        where(and_(funcionarios.c.cargo == 'Desenvolvedor Júnior')).
        values(salario=funcionarios.c.salario * 1.1)
    )

print('Salários atualizados!')

Salários atualizados!


**Passo 4**: Gere um relatório salarial agregando os dados. Construa um select combinando func.avg() para calcular a média salarial e group_by() para agrupar o resultado por cargo.

In [18]:
stmt = select(funcionarios.c.cargo, func.avg(funcionarios.c.salario)).group_by(funcionarios.c.cargo)
with engine.connect() as conn:
    result = conn.execute(stmt)

df_salarios = pd.DataFrame(result.fetchall(), columns=result.keys())
df_salarios

,cargo,avg_1
0,Analista,4000.0
1,Desenvolvedor,5000.0
2,Desenvolvedor Júnior,5000.0
3,Gerente,7000.0


##Nível 3: Avançado — ORM (Orientação a Objetos e Relacionamentos)

**Passo 1**: Transforme as tabelas em classes Python. Utilize declarative_base() e defina as classes Departamento e FuncionarioORM mapeando as colunas com mapped_column.

In [33]:
Base = declarative_base()

class Departamento(Base):
    __tablename__ = 'departamentos'
    id = mapped_column(Integer, primary_key=True)
    nome = mapped_column(String(50))
    funcionarios = relationship('FuncionarioORM', back_populates='departamento')

class FuncionarioORM(Base):
    __tablename__ = 'funcionarios'
    id = mapped_column(Integer, primary_key=True)
    nome = mapped_column(String(50))
    departamento_id = Column(Integer, ForeignKey('departamentos.id'))
    departamento = relationship('Departamento', back_populates='funcionarios')

Base.metadata.create_all(engine)
print('Tabelas criadas!')

Tabelas criadas!


**Passo 2**: Estabeleça a relação entre as classes configurando uma ForeignKey na tabela de funcionários e utilizando a função relationship em ambas as classes para permitir a navegação de objeto para objeto.

Feito acima!

**Passo 3**: Crie uma fábrica de sessões com sessionmaker(bind=engine) e instancie uma Session. Crie um objeto da classe Departamento e adicione funcionários a ele. Persista todos no banco de dados de uma só vez utilizando sessao.add() e sessao.commit().

In [34]:
Session = sessionmaker(bind=engine)
sessao = Session()

departamento = Departamento(nome='TI', funcionarios=[])
funcionario1 = FuncionarioORM(nome='João', departamento=departamento)
funcionario2 = FuncionarioORM(nome='Maria', departamento=departamento)
departamento.funcionarios.append(funcionario1)
departamento.funcionarios.append(funcionario2)
sessao.add(departamento)
sessao.commit()
print('Departamento e funcionários criados!')

Departamento e funcionários criados!


**Passo 4**: Faça uma consulta orientada a objetos para listar todos os funcionários do departamento de "TI". Utilize sessao.execute(select(...)).scalars().all() para que o SQLAlchemy retorne os objetos instanciados da classe em vez de linhas textuais. No final, utilize sessao.close() para fechar a sessão adequadamente.

In [38]:
funcionario_orm = sessao.execute(select(FuncionarioORM)).scalars().all()
sessao.close()